# 03 · Model Benchmark

Compare Ridge / XGBoost / LightGBM per position using blocked time-series CV (one season held out per fold). The ensemble picks different winners per position — GKP is nearly flat, FWD is variance-driven.

In [ ]:
from gaffer.providers.historical_csv import HistoricalCsvProvider
from gaffer.providers.fpl_api import LiveFplApiProvider
from gaffer.services.prediction_service import build_training_set
from gaffer.models.ridge_model import RidgePredictor
from gaffer.models.xgboost_model import XgbPredictor
from gaffer.models.lightgbm_model import LgbmPredictor
from gaffer.models.benchmark import benchmark_predictors

td = build_training_set(LiveFplApiProvider(), HistoricalCsvProvider())
X = td.X.dropna()
y = td.y.loc[X.index]
print('Training set:', X.shape)

In [ ]:
candidates = {
    'ridge': RidgePredictor(),
    'xgb': XgbPredictor(n_estimators=200, learning_rate=0.05),
    'lgbm': LgbmPredictor(n_estimators=300, learning_rate=0.05),
}
results = benchmark_predictors(candidates, X, y, td.seasons.loc[X.index])
results

In [ ]:
results.pivot(index='model', columns='fold', values='rmse').plot.bar()

**Next:** `04_uncertainty_quantification.ipynb` — fit a quantile regressor for 80% intervals.